# VocEd — Machine Learning Approaches: Logistic Regression, Random Forest, MLP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/Machine%20Learning%20Approaches.ipynb)

In Lab 04 we segmented urothelial-cell images by treating every pixel as a 3-D point in
RGB colour space and classifying it with **k-Nearest Neighbours**.  This lab is a
**complement** to Lab 04: same data, same train/test split, same flattened-pixel feature
matrix — but we swap k-NN for three other classical machine-learning methods.

We will train and evaluate:

- **Logistic Regression** — a linear classifier.  Learns three weighted sums of
  `(R, G, B)` (one per class) and picks the class with the highest score.  Decision
  boundaries are *planes* in 3-D colour space.
- **Random Forest** — an ensemble of decision trees.  Non-linear, flexible, and produces
  feature-importance scores for free.
- **MLP (Multi-Layer Perceptron)** — the simplest neural network: one hidden layer with
  a non-linear activation.

By the end you will be able to:

- Train sklearn's `LogisticRegression`, `RandomForestClassifier`, and `MLPClassifier`
  on the same pixel feature matrix.
- Compare their test-set Dice scores against k-NN from Lab 04.
- Argue whether the bottleneck in pixel-only segmentation is the **classifier** or the
  **feature representation**.


## 0. Setup


In [ ]:
!git clone https://github.com/emilsar/VocEd.git
%cd VocEd


## 1. Load Data & Recreate the Train/Test Split

Same dataset and split as Lab 04 (`np.random.seed(42)`), so the test-set Dice scores
here are directly comparable to the k-NN result from Lab 04.


In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

N = len(glob.glob('imagedata/X/*.npy'))
X = np.stack([np.load(f'imagedata/X/{i}.npy') for i in range(N)])   # (200, 3, 256, 256)
y = np.stack([np.load(f'imagedata/y/{i}.npy') for i in range(N)])   # (200, 256, 256)

# Same reproducible split as Lab 04
np.random.seed(42)
idx       = np.random.permutation(N)
train_idx = idx[:160]
test_idx  = idx[160:]

mask_cmap = ListedColormap(['black', 'steelblue', 'crimson'])
legend_patches = [
    mpatches.Patch(color='black',     label='0 — background'),
    mpatches.Patch(color='steelblue', label='1 — cytoplasm'),
    mpatches.Patch(color='crimson',   label='2 — nucleus'),
]
print(f'Loaded {N} images.  Train: {len(train_idx)}  Test: {len(test_idx)}')


## 2. Build the Per-Pixel Training Matrix

Identical to Lab 04: subsample 150 pixels per class per training image.  With 160
training images and 3 classes that gives `160 × 3 × 150 = 72,000` rows in a 3-column
matrix `(R, G, B)`.


In [ ]:
# ── Stratified subsample: equal pixels per class per training image ─────────
PIXELS_PER_CLASS = 150
rng = np.random.default_rng(42)

px_list, lbl_list = [], []
for i in train_idx:
    pixels = X[i].transpose(1, 2, 0).reshape(-1, 3)   # (65536, 3)
    labels = y[i].reshape(-1)                          # (65536,)
    for cls in [0, 1, 2]:
        cls_idx = np.where(labels == cls)[0]
        n       = min(PIXELS_PER_CLASS, len(cls_idx))
        chosen  = rng.choice(cls_idx, n, replace=False)
        px_list.append(pixels[chosen])
        lbl_list.append(labels[chosen])

X_train_px = np.vstack(px_list)
y_train_px = np.hstack(lbl_list)

print(f'Training pixel matrix : {X_train_px.shape}')
print(f'Training label vector : {y_train_px.shape}')
for cls, name in enumerate(['background', 'cytoplasm', 'nucleus']):
    n = (y_train_px == cls).sum()
    print(f'  {name:>12s}: {n:6d}  ({100*n/len(y_train_px):.1f}%)')


### Helpers we will reuse for every classifier

`predict_mask` flattens an image, runs `classifier.predict`, and reshapes the result
back to `(H, W)`.  `dice_score` measures overlap between predicted and ground-truth
masks for one class.  `mean_test_dice` averages cytoplasm and nucleus Dice across the
40 test images.


In [ ]:
def predict_mask(img, classifier):
    H, W = img.shape[1], img.shape[2]
    pixels = img.transpose(1, 2, 0).reshape(-1, 3)
    return classifier.predict(pixels).reshape(H, W)


def dice_score(pred, target, cls):
    pred_mask   = (pred   == cls)
    target_mask = (target == cls)
    intersection = (pred_mask & target_mask).sum()
    denom = pred_mask.sum() + target_mask.sum()
    return 1.0 if denom == 0 else 2 * intersection / denom


def mean_test_dice(classifier, name):
    scores, preds = [], {}
    for i in test_idx:
        p = predict_mask(X[i], classifier)
        preds[i] = p
        d = (dice_score(p, y[i], 1) + dice_score(p, y[i], 2)) / 2
        scores.append(d)
    print(f'{name} test Dice: {np.mean(scores):.4f}  ±  {np.std(scores):.4f}')
    return preds, scores


## 3. Logistic Regression

`LogisticRegression` learns three weighted sums of `(R, G, B)` — one per class — and
picks the class with the highest score (after a softmax).  Geometrically, the boundary
between any two classes is a *plane* in 3-D colour space.

It is the simplest baseline beyond k-NN and trains in less than a second on 72,000
pixels.


In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, n_jobs=-1)
print('Fitting Logistic Regression...')
lr.fit(X_train_px, y_train_px)
print('Done.')

# Visualise prediction on one test image
sample = test_idx[0]
pred_lr = predict_mask(X[sample], lr)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(X[sample].transpose(1, 2, 0))
axes[0].set_title(f'Image {sample} — RGB');         axes[0].axis('off')
axes[1].imshow(y[sample],  cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
axes[1].set_title('Ground truth');                   axes[1].axis('off')
axes[2].imshow(pred_lr, cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
axes[2].set_title('Logistic Regression prediction'); axes[2].axis('off')
fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, 0.0))
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()


## 4. Random Forest

A **forest** of decision trees.  Each tree is trained on a bootstrap sample of the data
with a random subset of features at every split; at prediction time the trees vote.

Random forests are non-linear: they can carve RGB space into arbitrarily many
axis-aligned boxes.  As a bonus, sklearn exposes **feature importances** — the average
contribution of each input dimension (R, G, B) to the splits — which we plot below.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
print('Fitting Random Forest...')
rf.fit(X_train_px, y_train_px)
print('Done.')

# ── Feature importance — only 3 features (R, G, B) ──────────────────────────
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['Red', 'Green', 'Blue'], rf.feature_importances_,
       color=['crimson', 'seagreen', 'steelblue'])
ax.set_ylabel('Feature importance')
ax.set_title('Random Forest — RGB feature importances')
plt.tight_layout()
plt.show()


In [ ]:
# Visualise prediction on the same test image
pred_rf = predict_mask(X[sample], rf)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(X[sample].transpose(1, 2, 0))
axes[0].set_title(f'Image {sample} — RGB');     axes[0].axis('off')
axes[1].imshow(y[sample],  cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
axes[1].set_title('Ground truth');               axes[1].axis('off')
axes[2].imshow(pred_rf, cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
axes[2].set_title('Random Forest prediction');   axes[2].axis('off')
fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, 0.0))
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()


## 5. MLP — Multi-Layer Perceptron

The simplest neural network.  One **hidden layer** of 16 neurons, each computing
`ReLU(w · pixel + b)`, followed by a softmax output that produces three class scores.
With 3 input features, 16 hidden units, and 3 outputs the MLP has only
`(3+1)×16 + (16+1)×3 = 115` learnable weights — yet it can carve smooth, curved
decision boundaries in RGB space.

Chapter 6 of the cvmath.club textbook unpacks how training works (gradient descent,
backpropagation, activation functions); here we treat it as a black-box classifier and
hand it the same `(72000, 3)` matrix.  Lab 05 onwards goes further: instead of one
weight per input pixel, we use *convolutions* that share weights across the image.


In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes=(16,), max_iter=200, random_state=42)
print('Fitting MLP (one hidden layer of 16 neurons, ReLU activation)...')
mlp.fit(X_train_px, y_train_px)
print('Done.')

# Visualise prediction on the same test image
pred_mlp = predict_mask(X[sample], mlp)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(X[sample].transpose(1, 2, 0))
axes[0].set_title(f'Image {sample} — RGB'); axes[0].axis('off')
axes[1].imshow(y[sample],  cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
axes[1].set_title('Ground truth');           axes[1].axis('off')
axes[2].imshow(pred_mlp, cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
axes[2].set_title('MLP prediction');         axes[2].axis('off')
fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, 0.0))
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()


## 6. Side-by-Side Comparison

Now run each classifier on the *full* 40-image test set and report the mean Dice score.
We also quote the k-NN result from Lab 04 for context.


In [ ]:
print('Evaluating all three classifiers on the full test set...\n')
preds_lr,  dice_lr  = mean_test_dice(lr,  'Logistic Regression')
preds_rf,  dice_rf  = mean_test_dice(rf,  'Random Forest      ')
preds_mlp, dice_mlp = mean_test_dice(mlp, 'MLP                ')

# k-NN result quoted from Lab 04 (k = 5, RGB-only).  Re-run Lab 04 to verify.
KNN_LAB04_DICE = 0.72

print('\nSummary (mean test Dice):')
print('=' * 50)
print(f'  k-NN (Lab 04, k=5)       {KNN_LAB04_DICE:.2f}   ← reference')
print(f'  Logistic Regression      {np.mean(dice_lr):.2f}')
print(f'  Random Forest            {np.mean(dice_rf):.2f}')
print(f'  MLP (16 hidden units)    {np.mean(dice_mlp):.2f}')
print('=' * 50)


In [ ]:
# ── Bar chart: mean test Dice for each classifier ──────────────────────────
methods = ['k-NN\n(Lab 04)', 'Logistic\nRegression', 'Random\nForest', 'MLP']
means   = [KNN_LAB04_DICE, np.mean(dice_lr), np.mean(dice_rf), np.mean(dice_mlp)]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(methods, means, color=['gray', 'steelblue', 'seagreen', 'crimson'])
ax.set_ylabel('Mean test Dice')
ax.set_ylim(0, 1.0)
ax.set_title('Pixel-only classifiers on RGB features')
for bar, v in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.015, f'{v:.2f}',
            ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# ── 4-way side-by-side prediction map for ONE test image ───────────────────
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(X[sample].transpose(1, 2, 0))
axes[0].set_title(f'Image {sample} — RGB');    axes[0].axis('off')
axes[1].imshow(y[sample], cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
axes[1].set_title('Ground truth');              axes[1].axis('off')
axes[2].imshow(preds_lr[sample],  cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
axes[2].set_title('Logistic Regression');       axes[2].axis('off')
axes[3].imshow(preds_rf[sample],  cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
axes[3].set_title('Random Forest');             axes[3].axis('off')
axes[4].imshow(preds_mlp[sample], cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
axes[4].set_title('MLP');                       axes[4].axis('off')
fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, 0.0))
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()


## 7. The Real Lesson — It's the Features, Not the Classifier

Look at the bar chart.  Four very different classifiers — a memory-based vote (k-NN), a
linear model (LR), an ensemble of trees (RF), and a small neural network (MLP) — all
score within a couple of Dice points of each other on this task.

This is **not** a coincidence.  Every one of them sees the same 3-D RGB vector for each
pixel and nothing else.  No matter how flexible the model is, it cannot reconstruct
information that was never given to it: whether a pixel sits on a cell boundary, inside
textured chromatin, or in smooth cytoplasm.  Those are properties of *neighbourhoods*,
not of single pixels.

**The bottleneck is the feature representation, not the classifier.**

What unlocks the next jump in Dice is changing the input — letting the classifier see a
small *patch* of pixels around each location instead of just the centre pixel.  That is
exactly what convolutional filters do, and it is the topic of:

- **Lab 05 — Convolutions**, where we hand-craft 3×3 kernels that produce richer
  per-pixel features (edges, blurs, textures).
- **Chapter 7 of the cvmath.club textbook**, which introduces feature engineering with
  Sobel, Gabor, and GLCM filters.
- **Chapter 8 of the textbook + Lab 06**, where a small CNN *learns* the kernels from
  data and finally pushes test Dice to ≈ 0.84.


---

## Group Exercise — Stretching Each Classifier

Each method has a knob you can turn.  Each person picks one and reports the test Dice
on `test_idx[:5]` (5 images, fast).  Compare results in your group.

| Person | Method | Knob to vary |
|---|---|---|
| A | Random Forest | `n_estimators=10`  vs `100`  vs `500` |
| B | MLP           | `hidden_layer_sizes=(4,)`  vs `(16,)`  vs `(64,)` |
| C | MLP           | `hidden_layer_sizes=(16,)`  vs `(16, 16)`  vs `(16, 16, 16)` |

Discuss:

- Does turning the knob change the Dice meaningfully?
- If not, what does that tell you about how much *signal* there is in the
  3-D RGB feature?
- What change to the **inputs** (not the model) would make the biggest difference?
